# 04 — Panel Scoring (Pre / Post Windows)

Scores each panel user's text in the **pre-baseline** and **post-outcome** windows,
then merges with exposure labels to build the analysis panel.

**Pre-period definition (Sep–Nov, before first anchor comment):**
- Exposed users: all Sep–Nov posts/comments *before* their first anchor comment
- Unexposed users: all Sep–Nov posts/comments (no anchor comment, so full Sep–Nov)
- This replaces the fixed August window to maximise coverage (~36% vs ~7%)

**Post-period:** December–May of each cycle year

**Inputs:**
- `cleaned_output/r_gradadmissions_posts.cleaned.jsonl` + `Grad Admissions Comments.jsonl`
- `data/processed/exposure_labels.parquet` (from notebook 03)
- `data/processed/anchor_posts.parquet` (from notebook 03)

**Outputs:**
- `data/processed/panel_scores.parquet`
- `data/processed/post_level_scores.parquet`
- `data/processed/dose_exposure.parquet`


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                     STUDY CONFIG                            ║
# ║  Keep CYCLES in sync with NB03 CONFIG                       ║
# ╚══════════════════════════════════════════════════════════════╝

STUDY_ID  = 'gradadmissions'
SUBREDDIT = 'gradadmissions'   # change to 'mscs' for MSCS pipeline

# Must match NB03 CYCLES exactly. Numbered chronologically:
#   Cycle 1: Aug 2022 – May 2023 admission cycle
#   Cycle 2: Aug 2023 – May 2024 admission cycle
#   Cycle 3: Aug 2024 – May 2025 admission cycle
CYCLES = {
    1: {
        'anchor_start': '2022-09-01', 'anchor_end': '2022-11-30',
        'post_start':   '2022-12-01', 'post_end':   '2023-05-31',
    },
    2: {
        'anchor_start': '2023-09-01', 'anchor_end': '2023-11-30',
        'post_start':   '2023-12-01', 'post_end':   '2024-05-31',
    },
    3: {
        'anchor_start': '2024-09-01', 'anchor_end': '2024-11-30',
        'post_start':   '2024-12-01', 'post_end':   '2025-05-31',
    },
}

# Pre-period strategy:
#   'anchor_window_before_first_comment' — Sep-Nov before first anchor comment (~35% coverage)
#   'fixed_month'                        — fixed calendar month (set FIXED_PRE_MONTH)
PRE_PERIOD_STRATEGY = 'anchor_window_before_first_comment'
FIXED_PRE_MONTH     = 8   # only used if PRE_PERIOD_STRATEGY == 'fixed_month'


In [2]:
import json
import numpy as np
import pandas as pd
import joblib
from datetime import datetime, timezone
from pathlib import Path

ROOT      = Path('..').resolve()
DATA   = ROOT / 'data' / 'processed' / SUBREDDIT
MODEL_DIR = ROOT / 'models'

POSTS_CLEAN    = DATA / 'posts_clean.jsonl'
COMMENTS_CLEAN = DATA / 'comments_clean.jsonl'
EXPOSURE_PATH = DATA / 'exposure_labels.parquet'
ANCHOR_PATH   = DATA / 'anchor_posts.parquet'
OUT_PATH      = DATA / 'panel_scores.parquet'

# Parse CYCLES dates to datetime objects
_CYCLES_DT = {}
for c, w in CYCLES.items():
    _CYCLES_DT[c] = {
        'anchor_start': datetime.fromisoformat(w['anchor_start']).replace(tzinfo=timezone.utc),
        'anchor_end':   datetime.fromisoformat(w['anchor_end'] + 'T23:59:59').replace(tzinfo=timezone.utc),
        'post_start':   datetime.fromisoformat(w['post_start']).replace(tzinfo=timezone.utc),
        'post_end':     datetime.fromisoformat(w['post_end'] + 'T23:59:59').replace(tzinfo=timezone.utc),
    }

print(f'Study: {STUDY_ID} | Pre-period strategy: {PRE_PERIOD_STRATEGY}')
print('Paths OK:', all(p.exists() for p in [POSTS_CLEAN, COMMENTS_CLEAN, EXPOSURE_PATH, ANCHOR_PATH]))


Study: mba | Pre-period strategy: anchor_window_before_first_comment
Paths OK: True


## 1) Load panel users

In [3]:
exposure = pd.read_parquet(EXPOSURE_PATH)
print(f'Panel users: {exposure["author"].nunique():,}  |  rows: {len(exposure):,}')
print(exposure['exposed'].value_counts())
panel_users = set(exposure['author'])

anchor_posts_df = pd.read_parquet(ANCHOR_PATH, columns=['id', 'cycle'])
anchor_ids = set(anchor_posts_df['id'].astype(str))
print(f'Anchor post IDs: {len(anchor_ids):,}')


Panel users: 23,253  |  rows: 25,139
exposed
False    21377
True      3762
Name: count, dtype: int64
Anchor post IDs: 471


## 2) Find first anchor comment per exposed user


In [ ]:
# Scan raw comments to find each exposed user's first anchor comment timestamp
# Pre-period cutoff = this timestamp; activity before it counts as pre-period
print('Scanning comments for first anchor comment per exposed user...')
first_anchor_comment = {}  # author -> datetime

with open(COMMENTS_CLEAN) as f:
    for line in f:
        line = line.strip()
        if not line: continue
        r = json.loads(line)
        author  = r.get('author')
        post_id = r.get('post_id', '')
        if author not in panel_users or post_id not in anchor_ids: continue
        dt = datetime.fromisoformat(r['created_dt'])
        if author not in first_anchor_comment or dt < first_anchor_comment[author]:
            first_anchor_comment[author] = dt

print(f'First anchor comment found for {len(first_anchor_comment):,} exposed users')


In [ ]:
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')
print('Classifiers loaded.')

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def score_texts_all(texts):
    if not texts:
        return np.array([]), np.array([]), np.array([]), np.array([])
    anx  = sigmoid(clf_anx.decision_function(texts))
    dep  = sigmoid(clf_dep.decision_function(texts))
    str_ = sigmoid(clf_str.decision_function(texts))
    mean = np.stack([anx, dep, str_], axis=1).mean(axis=1)
    return anx, dep, str_, mean


## 4) Build corpus and score


In [ ]:
def assign_window(author, dt, cycle):
    w = _CYCLES_DT[cycle]
    if w['post_start'] <= dt <= w['post_end']:
        return 'post'
    if w['anchor_start'] <= dt <= w['anchor_end']:
        if PRE_PERIOD_STRATEGY == 'anchor_window_before_first_comment':
            cutoff = first_anchor_comment.get(author)
            if cutoff is None or dt < cutoff:
                return 'pre'
        elif PRE_PERIOD_STRATEGY == 'fixed_month':
            if dt.month == FIXED_PRE_MONTH:
                return 'pre'
    return None

user_cycles = exposure.groupby('author')['cycle'].apply(list).to_dict()

records = []

def process_file(path, text_field):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            r = json.loads(line)
            author = r.get('author')
            if author not in panel_users: continue
            dt = datetime.fromisoformat(r['created_dt'])
            text = r.get(text_field, '') or ''
            for cycle in user_cycles.get(author, []):
                window = assign_window(author, dt, cycle)
                if window:
                    records.append({'author': author, 'cycle': cycle, 'window': window,
                                    'created_dt': dt.isoformat(), 'clean_text': text})

print('Processing posts...')
process_file(POSTS_CLEAN, 'clean_text')
print('Processing comments...')
process_file(COMMENTS_CLEAN, 'clean_text')

corpus = pd.DataFrame(records)
print(f'\nWindow distribution:')
print(corpus.groupby(['cycle','window']).size())
print(f'Unique authors: {corpus["author"].nunique():,}')


## 4a) Score all records


In [ ]:
texts = corpus['clean_text'].tolist()
print(f'Scoring {len(texts):,} records...')
corpus['anx_score'], corpus['dep_score'], corpus['str_score'], corpus['mean_mh_score'] = score_texts_all(texts)
print('Done.')
corpus[['anx_score', 'dep_score', 'str_score', 'mean_mh_score']].describe().round(4)

## 4b) Save post-level scores (for post-level DiD in NB06)

In [ ]:
# Save individual post/comment records with scores before aggregation.
# Used by NB06 for post-level DiD (recovers ~147K observations vs 1,094 user-means).
POST_LEVEL_PATH = DATA / 'post_level_scores.parquet'

post_level = corpus[['author', 'cycle', 'window', 'created_dt',
                      'anx_score', 'dep_score', 'str_score', 'mean_mh_score']].copy()
post_level['created_dt'] = pd.to_datetime(post_level['created_dt'], utc=True)

post_level.to_parquet(POST_LEVEL_PATH, index=False)
print(f'Saved {len(post_level):,} post-level rows → {POST_LEVEL_PATH}')
print(post_level.groupby(['cycle', 'window']).size())

## 4c) Compute dose-response data (# anchor-thread comments per user)

In [ ]:
import json as _json

ANCHOR_PATH = DATA / 'anchor_posts.parquet'
DOSE_PATH   = DATA / 'dose_exposure.parquet'

anchor_posts_df = pd.read_parquet(ANCHOR_PATH, columns=['id', 'cycle'])
anchor_ids      = set(anchor_posts_df['id'].astype(str))

# Load only comments that are on anchor threads
dose_rows = []
with open(COMMENTS_CLEAN) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = _json.loads(line)
        if r.get('post_id') in anchor_ids and r.get('author') in panel_users:
            dose_rows.append({'author': r['author'], 'post_id': r['post_id']})

dose_comments = pd.DataFrame(dose_rows)

# Map post_id → cycle
pid_to_cycle = anchor_posts_df.set_index('id')['cycle'].to_dict()
dose_comments['cycle'] = dose_comments['post_id'].map(pid_to_cycle)

# Count comments per (author, cycle)
dose = (
    dose_comments.groupby(['author', 'cycle'])
    .size()
    .reset_index(name='n_anchor_comments')
)
dose['log1p_n_anchor'] = np.log1p(dose['n_anchor_comments'])

dose.to_parquet(DOSE_PATH, index=False)
print(f'Saved {len(dose):,} dose records → {DOSE_PATH}')
print(dose['n_anchor_comments'].describe().round(2))

## 5) Aggregate per (author, cycle, window)

In [ ]:
agg = (
    corpus
    .groupby(['author', 'cycle', 'window'])
    .agg(
        mh_score  = ('mean_mh_score', 'mean'),
        anx_score = ('anx_score',     'mean'),
        dep_score = ('dep_score',     'mean'),
        str_score = ('str_score',     'mean'),
        n_posts   = ('mean_mh_score', 'count'),
    )
    .reset_index()
)

def pivot_window(window_label, prefix):
    sub = agg[agg['window'] == window_label].drop(columns='window')
    return sub.rename(columns={
        'mh_score':  f'{prefix}_mh_score',
        'anx_score': f'{prefix}_anx_score',
        'dep_score': f'{prefix}_dep_score',
        'str_score': f'{prefix}_str_score',
        'n_posts':   f'{prefix}_n_posts',
    })

pre  = pivot_window('pre',  'pre')
post = pivot_window('post', 'post')

scores = pre.merge(post, on=['author', 'cycle'], how='inner')
print(f'Users with both pre and post observations: {len(scores):,}')

## 6) Merge with exposure labels

In [ ]:
panel = exposure.merge(scores, on=['author', 'cycle'], how='inner')
print(f'Final panel rows: {len(panel):,}')
print(f'Unique users:     {panel["author"].nunique():,}')
print(f'\nCoverage: {100 * panel["author"].nunique() / len(panel_users):.1f}% of panel users have pre+post scores')
print('\nExposure breakdown:')
print(panel.groupby(['cycle', 'exposed']).size())

In [ ]:
# Score distribution check
print('Pre-period MH scores:')
print(panel.groupby('exposed')['pre_mh_score'].describe().round(4))
print('\nPost-period MH scores:')
print(panel.groupby('exposed')['post_mh_score'].describe().round(4))

## 7) Save

> **Approval gate:** Review coverage stats and score distributions above before running this cell.

In [ ]:
out_cols = [
    'author', 'cycle', 'exposed', 'exposure_intensity',
    'pre_mh_score',  'pre_anx_score',  'pre_dep_score',  'pre_str_score',  'pre_n_posts',
    'post_mh_score', 'post_anx_score', 'post_dep_score', 'post_str_score', 'post_n_posts',
]
panel[out_cols].to_parquet(OUT_PATH, index=False)
print(f'Saved {len(panel):,} rows → {OUT_PATH}')
print('Columns:', out_cols)
print('\nexposure_intensity distribution:')
print(panel.groupby(['cycle','exposure_intensity']).size().unstack(fill_value=0))
